> ### Note on Labs and Assignments:
>
> 🔧 Look for the **wrench emoji** 🔧 — it highlights where you're expected to take action!
>
> These sections are graded and are not optional.
>

# IS 4487 Lab 14: API Integration

## Outline
1. Import customer reviews
2. Create prompts for LLM
3. Summarize Customer Reviews

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Labs/lab_14_api.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Customer Reviews - Business Context

This dataset represents customer reviews of Megatelco phones, which provides valuable insight into customer satisfaction, product performance, and brand perception. In a business context, companies use this type of data to understand how customers feel about their products, identify common issues (like battery life or software bugs), and detect what features customers value most (such as camera quality or speed). Because reviews are unstructured text, they contain rich, detailed feedback that goes beyond simple ratings—making them especially useful for improving products, guiding marketing strategies, and reducing customer churn.

In this lab, instead of using traditional text analysis methods, you will use a large language model (LLM) to analyze the reviews. Traditional text analytics techniques (like keyword counting or basic sentiment tools such as VADER) rely on predefined rules or dictionaries and often struggle with nuance, sarcasm, or context. LLMs, on the other hand, are trained on massive amounts of text and can understand language more like a human—recognizing tone, context, and subtle meaning. This allows them to perform more advanced tasks, such as summarizing feedback, detecting themes, interpreting sarcasm, and generating insights directly from raw text. Using an LLM enables businesses to extract deeper, more accurate insights from customer reviews at scale, making it a powerful tool for modern data analysis.

## Data Dictionary

| Column                        | Data Type       | Description                                                  |
|------------------------------|------------------|--------------------------------------------------------------|
| `Date`                   | Date           | Date of the review                                              |
| `Stars`                 | Integer           | Number of stars, from 1 (low) to 5 (high)                                     |
| `Review`             | String       | Text of the customer review                      |


## APIs for AI

An API for a large language model (LLM) allows applications to send text (like a question or prompt) to a powerful AI model hosted on a server and receive a generated response in return. Instead of building and running their own AI models, businesses can simply make requests to the API—often with just a few lines of code—and get capabilities like text generation, summarization, classification, or customer support automation.

This is useful for businesses because it makes advanced AI accessible without requiring deep technical expertise or expensive infrastructure. Companies can quickly integrate LLMs into their products and workflows to improve efficiency, enhance customer experiences (like chatbots or personalized responses), and gain insights from large amounts of text data, all while saving time and development costs.




## Part 1: Load the Data

### What you are going to do:
- Load the dataset
- Preview the data

**Things to notice:**
- Do you see any elements in the reviews that would difficult for VADER or other lexicon-based models to process?


In [22]:
import pandas as pd
import google.generativeai as genai

Create a dataframe containing sample reviews for use in the lab

In [21]:
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/megatelco_customer_reviews.csv"
df = pd.read_csv(url)

df.head()

,Date,Stars,Review
0,2025-12-01,4,"I purchased a Megatelco phone last week, and i..."
1,2025-12-03,3,"My Megatelco phone arrived fast, but I've noti..."
2,2025-12-05,5,Megatelco offers top-notch phones! I'm happy w...
3,2025-12-08,4,I've been using Megatelco phones for a while n...
4,2025-12-10,2,"Regrettably, my experience with Megatelco was ..."


# Part 2 : Prepare the LLM Prompt
### What you are going to do:
- Instruct the LLM on the context and desired output

### Why this matters:
A LLM needs to know what you are trying to accomplish, what data you will provide, what it should do with the data, and how to format the output.  Your prompt needs to set all of this context before passing in a review.  

In [20]:
base_prompt = (
    "Summarize the sentiment and most important points in the following user review for "
    "a phone company named Megatelco.  You will evaluate the sentiment, theme (in 2 words or less), count the number of words, and suggest an action (in 2 words or less)."
     "Format the output in a table with columns: "
     "Sentiment, Theme, Word Count, Suggested Action. Review: "
)

#Get the first review and pair it with the prompt
review = df['Review'].values[0]
prompt = base_prompt + review
print(prompt)

Summarize the sentiment and most important points in the following user review for a phone company named Megatelco.  You will evaluate the sentiment, theme (in 2 words or less), count the number of words, and suggest an action (in 2 words or less).Format the output in a table with columns: Sentiment, Theme, Word Count, Suggested Action. Review: I purchased a Megatelco phone last week, and it has sick performance. The camera quality is great, and the battery life is long. Overall, a solid 4-star experience.


### 🔧 Try It Yourself — Part 2

1. Create a new version of the prompt that adds two additional columns to the output.   (this columns should be numeric values you can visualize at a later time)

### In Your Response:
1. Why did you pick the two columns that you added?  What business insight would they provide?

In [23]:
new_base_prompt = (
    "Summarize the sentiment and most important points in the following user review for "
    "a phone company named Megatelco.  You will evaluate the sentiment, theme (in 2 words or less), count the number of words, suggest an action (in 2 words or less), count the number of sentences, and assign a sentiment score from 1 to 10."
     "Format the output in a table with columns: "
     "Sentiment, Theme, Word Count, Suggested Action, Sentence Count, Sentiment Score (1-10). Review: "
)

#Get the first review and pair it with the prompt
review = df['Review'].values[0]
new_prompt = new_base_prompt + review
print(new_prompt)

Summarize the sentiment and most important points in the following user review for a phone company named Megatelco.  You will evaluate the sentiment, theme (in 2 words or less), count the number of words, suggest an action (in 2 words or less), count the number of sentences, and assign a sentiment score from 1 to 10.Format the output in a table with columns: Sentiment, Theme, Word Count, Suggested Action, Sentence Count, Sentiment Score (1-10). Review: I purchased a Megatelco phone last week, and it has sick performance. The camera quality is great, and the battery life is long. Overall, a solid 4-star experience.


### ✍️ Your Response: 🔧
1. I chose to include sentence count and sentiment score as the new vairables as they measure both review detail and customer sentiment associated with the score.

# Part 3: Connect with the API and Test

### What you are going to do:
- Create a connection to Gemini
- Run a test prompt
- Pass the full collection of reviews to the API (either in a batch or one-by-one in a loop)
- Format the output in a dataframe.   

### Do the following
- Go to https://aistudio.google.com/api-keys
- Click on the `Get API key` link on the bottom left corner
- Copy the value into the box below
- Send the first customer review to Gemini for analysis, then view the result

**Things to notice:**
- Is there any limit to the number of free requests you can make to Gemini?  (without payment)

In [24]:
# Configure the API key
API_KEY = 'secretName'
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('models/gemini-2.5-flash')
print("Gemini API configured successfully.")

Gemini API configured successfully.


In [25]:
response = model.generate_content(prompt)
print(response.candidates[0].content.parts[0].text)

BadRequest: 400 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.

### 🔧 Try It Yourself — Part 3
Ask Gemini to help you loop through the reviews, one by one, and format them into a dataframe. Use the following steps:
1. Build a full prompt by combining the base prompt that you created above with one or more reviews
2. Pass the full prompt to Gemini
3. Format the response into a dataframe
4. If you are processing one row at a time, pass the next prompt (in a loop) until you have processed all of the reviews
5. Show the final dataframe using `df.head()`

### In Your Response:
1. How does the output of the LLM compare to the output we saw in week 13 from VADER or TextBlob?

In [ ]:
# 🔧 Add code here

### ✍️ Your Response: 🔧
1.



# Part 4: Visualize the Output
### What you are going to do:
- create visualizations to summarize the customer reviews.

## Why this matters:
If we have thousands of reviews, you will need to summarize them for management use.  Each chart should tell a distinct story about the customer feedback, themes and suggested action items.  

### 🔧 Try It Yourself — Part 4
Create at least four visualizations to answer the following questions:
1. What are the main themes?
2. For each theme, what is the sentiment associated with the theme?
3. What are the action items that should be taken to reduce churn?
4. Add one or more visualizations that will show the insights from the fields that you added in part 2.  

### In Your Response:
1. Why did you pick the charts or image types for each of the four visualizations?  

In [ ]:
# 🔧 Enter your code here

### ✍️ Your Response: 🔧
1.


## 🔧 Part 5: Reflection (100 words or less)

In this lab you connected to an LLM API to request summarization of customer reviews.  

Use the cell below to answer the following questions:

1. What was the elapsed time to collect the LLM responses to all of the requests?  How long would it take to process 1,000 requests?
2. What are the advantages and disadvantes of using Gemini versus VADER or TextBlob, which we used in Lab 13?  
3. Write a prompt that you could use to an LLM to create a business strategy and business plan to improve customer churn.   

### ✍️ Your Response: 🔧
1.

2.

3.

# Export Your Notebook to Submit in Canvas
Use the instructions from Lab 1

In [ ]:
!jupyter nbconvert --to html "lab_14_api.ipynb"